In [ ]:
import kagglehub
import os
import pandas as pd
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1 Load the dataset

path_Q1 = os.path.join(path, 'Q1_data.csv')

df = pd.read_csv(path_Q1)
print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2:
df.head()

In [ ]:
# Task 3
df.info()

In [ ]:
# Task 4:
df.describe()

In [ ]:
# Task 5:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1:
df = df.drop(columns="Order_ID")
df

In [ ]:
# Task 2:
missing = pd.DataFrame(
    df.isnull().sum(), columns=['count'])
missing = missing[missing['count']>0]
missing

In [ ]:
# drop all missing label since it does not give us informtion in trainging
df_clean = df.dropna(subset=['Delivery_Time']).copy()
print(f"shape after removing unlabeled data {df_clean.shape}")
df_clean.head()

In [ ]:
set(df_clean['Weather'])

In [ ]:
df_clean['Weather'] = df_clean['Weather'].fillna(df_clean['Weather'].mode()[0])
set(df_clean['Weather'])

In [ ]:
set(df_clean['Traffic_Level'])

In [ ]:
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna(df_clean['Traffic_Level'].mode()[0])
set(df_clean['Traffic_Level'])

In [ ]:
set(df_clean['Time_of_Day'])

In [ ]:
df_clean['Time_of_Day'] = df_clean['Time_of_Day'].fillna(df_clean['Time_of_Day'].mode()[0])
set(df_clean['Time_of_Day'])

In [ ]:
set(df_clean['Courier_Experience_yrs'])

In [ ]:
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])
set(df_clean['Courier_Experience_yrs'])

In [ ]:
# check if missing some proccesd
missing = pd.DataFrame(
    df_clean.isnull().sum(), columns=['count'])
missing = missing[missing['count']>0]
missing

In [ ]:
# Task 3:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4:
from sklearn.preprocessing import LabelEncoder

cols =  df_clean.select_dtypes('object').columns
le = LabelEncoder()
for col in cols:
  df_clean[col] = le.fit_transform(df_clean[col])

df_clean.head()

In [ ]:
# Task 5:
from sklearn.preprocessing import StandardScaler
features = df_clean.columns.drop("Delivery_Time")

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
from sklearn.model_selection import train_test_split

# Task 1:
X = df_clean.drop(columns='Delivery_Time')
y = df_clean['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5:
from sklearn.model_selection import KFold
from sklearn.metrics import  mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

n_splits = 5

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

mae = []
model =  RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train

  model.fit(X_train,y_train)
  print('model traind')

  # Predict
  y_pred = model.predict(X_test)
  mae.append( mean_absolute_error(y_test, y_pred))


print(f"averaged score {sum(mae)/len(mae)}")


In [ ]:
# Task 1:

feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Delivery Time (Ground Truth)")
plt.ylabel("Predicted delivery Time")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(y_pred)

plt.title("Predicted Delivery Time hist")

plt.tight_layout()
plt.show()